# Vector stores and semantic search



In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

c:\Users\kevin\miniconda3\envs\deeplearning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self._model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self._model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        # Normalizamos
        norms = np.linalg.norm(new_embeddings, axis=1, keepdims=True)
        new_embeddings = new_embeddings / np.maximum(norms, 1e-10)

        self._documents.extend(documents)
        if self._embeddings is None:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self._embeddings is None or len(self._documents) == 0:
            return []

        query_emb = self._model.encode([query], convert_to_numpy=True)
        query_emb = query_emb / np.maximum(np.linalg.norm(query_emb, keepdims=True), 1e-10)

        scores = (self._embeddings @ query_emb.T).flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            SearchResult(score=float(scores[i]), document=self._documents[i])
            for i in top_indices
        ]

## Cargar el dataset

In [3]:
import pandas as pd

DATASET_URL = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"

df = pd.read_csv(DATASET_URL)
print(df.shape)
df.head()

(7734, 5)


,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [4]:
# Aquí creamos los documentos
documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name":     str(row.get("animal_name", "")),
            "source":          str(row.get("source", "")),
            "media_link":      str(row.get("media_link", "")),
            "wikipedia_link":  str(row.get("wikipedia_link", "")),
        },
    )
    for _, row in df.iterrows()
    if pd.notna(row.get("text")) and str(row.get("text", "")).strip()
]

## Indexing y consultas

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

vs = VectorStore(embedding_model=model)
vs.add_documents(documents)

print(f"VectorStore listo con {len(vs._documents)} documentos.")

Batches: 100%|██████████| 242/242 [00:02<00:00, 112.85it/s]

VectorStore listo con 7731 documentos.


In [6]:
def print_results(query: str, results: list[SearchResult]):
    print(f"Query: '{query}'")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        print(f"  #{i}  score={r.score:.4f}")
        print(f"       text    : {r.document.text[:120]}...")
        print(f"       metadata: {r.document.metadata}")
        print()

queries = [
    "animals that live in the ocean",
    "fastest animals on land",
    "animals with unique sleeping habits",
    "animals that use tools",
    "venomous or poisonous creatures",
]

for q in queries:
    results = vs.search(q, top_k=3)
    print_results(q, results)
    print("=" * 70)
    print()

Query: 'animals that live in the ocean'
----------------------------------------------------------------------
  #1  score=0.6498
       text    : Smallest cetacean in the ocean...
       metadata: {'animal_name': 'vaquita', 'source': 'https://a-z-animals.com/animals/vaquita/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Vaquita'}

  #2  score=0.6452
       text    : White Sharks live in all of the world's oceans....
       metadata: {'animal_name': 'white shark', 'source': 'https://a-z-animals.com/animals/white-shark/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Great_white_shark'}

  #3  score=0.6317
       text    : May eat squid or other small invertebrate ocean life...
       metadata: {'animal_name': 'bonito fish', 'source': 'https://a-z-animals.com/animals/bonito-fish/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Bonito'}


Query: 'fastest animals on land'
----------------------------------------------------------------------
  #1  score=0.8567
       text    : Fastest a

## Part II: Filtering by metadata

In [7]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self._model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None 

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self._model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        norms = np.linalg.norm(new_embeddings, axis=1, keepdims=True)
        new_embeddings = new_embeddings / np.maximum(norms, 1e-10)

        self._documents.extend(documents)
        if self._embeddings is None:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if self._embeddings is None or len(self._documents) == 0:
            return []

        # Determinar índices según el filtro
        if metadata_filter:
            indices = [
                i for i, doc in enumerate(self._documents)
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
        else:
            indices = list(range(len(self._documents)))

        if not indices:
            return []

        candidate_embeddings = self._embeddings[indices]

        query_emb = self._model.encode([query], convert_to_numpy=True)
        query_emb = query_emb / np.maximum(np.linalg.norm(query_emb, keepdims=True), 1e-10)

        scores = (candidate_embeddings @ query_emb.T).flatten()

        top_local = np.argsort(scores)[::-1][:top_k]
        return [
            SearchResult(score=float(scores[j]), document=self._documents[indices[j]])
            for j in top_local
        ]

## Dataset y queries

In [15]:
from sklearn.datasets import fetch_20newsgroups

# 20 Newsgroups: artículos de foros con 20 categorías (política, deportes, ciencia, tecnología, religión, etc.)
raw = fetch_20newsgroups(subset="test", remove=("headers", "footers", "quotes"))

news_documents = [
    Document(
        text=text.strip(),
        metadata={"category": raw.target_names[label]},
    )
    for text, label in zip(raw.data, raw.target)
    if text.strip()
]

print(f"Documentos cargados: {len(news_documents)}")
print(f"Categorías: {raw.target_names}")
print("\nEjemplo:")
print(f"  text    : {news_documents[0].text[:120]}...")
print(f"  metadata: {news_documents[0].metadata}")

Documentos cargados: 7317
Categorías: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

Ejemplo:
  text    : I am a little confused on all of the models of the 88-89 bonnevilles.
I have heard of the LE SE LSE SSE SSEI. Could some...
  metadata: {'category': 'rec.autos'}


In [16]:
fvs = FilteredVectorStore(embedding_model=model)
fvs.add_documents(news_documents)

print(f"FilteredVectorStore listo con {len(fvs._documents)} documentos.")

Batches: 100%|██████████| 229/229 [00:04<00:00, 57.16it/s] 

FilteredVectorStore listo con 7317 documentos.


In [17]:
filtered_queries = [
    ("NASA rocket launch moon",             {"category": "sci.space"}),
    ("Stanley Cup playoffs overtime goal",  {"category": "rec.sport.hockey"}),
    ("gun control legislation congress",    {"category": "talk.politics.guns"}),
    ("3D rendering software OpenGL",        {"category": "comp.graphics"}),
    ("used car price engine problems",      {"category": "rec.autos"}),
]

for query, f in filtered_queries:
    results = fvs.search(query, top_k=3, metadata_filter=f)
    print(f"Query : '{query}'  |  filter: {f}")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        print(f"  #{i}  score={r.score:.4f}")
        print(f"       text    : {r.document.text[:120]}...")
        print(f"       metadata: {r.document.metadata}")
        print()
    print("=" * 70)
    print()

Query : 'NASA rocket launch moon'  |  filter: {'category': 'sci.space'}
----------------------------------------------------------------------
  #1  score=0.4774
       text    : Cool!
	I think you mean Moon.
		(Sorry, I had to.)  ; )...
       metadata: {'category': 'sci.space'}

  #2  score=0.4211
       text    : I have often thought about, if its possible to have a powerfull laser
on earth, to light at the Moon, and show lasergrap...
       metadata: {'category': 'sci.space'}

  #3  score=0.4128
       text    : Original to: wats@scicom.AlphaCDC.COM
G'day wats@scicom.AlphaCDC.COM

20 Apr 93 18:17, wats@scicom.AlphaCDC.COM wrote to...
       metadata: {'category': 'sci.space'}


Query : 'Stanley Cup playoffs overtime goal'  |  filter: {'category': 'rec.sport.hockey'}
----------------------------------------------------------------------
  #1  score=0.6035
       text    : It's over - the Sabres came back to beat the Bruins in OT 6-5 tonight
to sweep the series. A beautiful goal by B

## Reflexiones

Los resultados fueron bastante buenos. Por ejemplo, la búsqueda semántica de fastest animals on land dieron conceptops relacionados, como el cheetah y el peregrine falcon. Del lado del FilteredVectorStore también se obtuvieron textos relacionados. Aunque los scores fueron un poco más bajos en sci.space, que aquí me imagino que fue porque los textos son más informales que los de los animales por ejemplo. Finalmente también quiero agregar que el filtro por metadatos nos garantiza que los resultados pertenezcan a una categoría específica. Se me ocurre que sin filtro, una query de una categoría podría devolver un artículo de otra si es que se mencionan palabras relacionadas.